In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import lbpscreen as lbp
from pathlib import Path

models = lbp.load_models("../data/processed/curated_models")
m = models["E_coli_Nissle1917"]

## growth_table

**Purpose**: Growth rate and solver status for every model in one table. The status column matters: 0.0 with status optimal is real zero growth, while nan with infeasible means the model is broken.

In [2]:
sol = m.optimize()
print(sol.objective_value)
print(sol.status)

93.91435011269724
optimal


In [3]:
for label, model in models.items():
    sol = model.optimize()
    print(label, round(sol.objective_value, 3), sol.status)

A_muciniphila_BAA835 40.778 optimal
E_coli_Nissle1917 93.914 optimal
E_faecalis_MMH594 76.88 optimal
E_lenta_DSM2243 59.292 optimal


In [4]:
lbp.growth_table(models)

,Objective,Growth,Status
Strain,,,
A_muciniphila_BAA835,biomass345,40.777835,optimal
E_coli_Nissle1917,bio1,93.914350,optimal
E_faecalis_MMH594,bio1,76.880376,optimal
E_lenta_DSM2243,biomass467,59.291906,optimal


## flux_range
**Purpose:** One FBA gives one flux solution, but many are often equally good. Flux variability analysis (FVA) reports the lowest and highest flux each reaction can carry while growth stays near its maximum. This tells you whether a reaction is forced (always active) or merely possible.

**Why fraction?**
FVA holds growth at a fraction of its maximum. For example, *fraction=0.9* means "explore all flux states where the cell still grows at >=90% of top speed".

In [5]:
from cobra.flux_analysis import flux_variability_analysis

fva = flux_variability_analysis(m, [m.reactions.EX_glc_D_e_], fraction_of_optimum=0.9)
fva

,minimum,maximum
EX_glc_D_e_,-1000.0,-7.958079e-13


In [6]:
# A forced reaction has minimum ~ maximum. That reaction must carry the flux
diff = (fva["maximum"] - fva["minimum"]).abs()
print(diff < 1e-6)

EX_glc_D_e_    False
dtype: bool


In [7]:
lbp.flux_range(m, ["EX_glc_D_e_", "EX_ac_e_"], fraction=0.9)   # glucose, acetate

,minimum,maximum,fixed
EX_glc_D_e_,-1000.0,0.0,False
EX_ac_e_,-1000.0,1000.0,False


In [8]:
gene = list(m.genes)[0]
with m:
    gene.knock_out()
    print(gene.id, "KO growth:", m.slim_optimize())
print("restored:", m.slim_optimize())

ECOLIN_RS20580 KO growth: 93.91435011269724
restored: 93.91435011269724


In [9]:
from cobra.flux_analysis import single_gene_deletion

res = single_gene_deletion(m)
res.head()

,ids,growth,status
0,{WP_000371928_1_B},93.91435,optimal
1,{WP_000786563_1},93.91435,optimal
2,{WP_001275885_1},93.91435,optimal
3,{ECOLIN_RS01570},93.91435,optimal
4,{WP_001186653_1},93.91435,optimal


In [10]:
ess = lbp.essential_genes(m)              # takes a few minutes on Nissle
print(ess["essential"].value_counts())    # how many essential vs not
ess[ess["essential"]].head(10)            # the essential genes, lowest growth first

essential
False    1321
True      136
Name: count, dtype: int64


,growth,wild_type,ko_fraction,essential
gene,,,,
ECOLIN_RS09460,-2.475156e-11,93.91435,-2.635546e-13,True
316435_3_622_1_peg,-7.399133e-12,93.91435,-7.878597e-14,True
316435_3_3011_6_peg,-5.780219e-12,93.91435,-6.154778e-14,True
316435_3_2438_4_peg,-3.258642e-12,93.91435,-3.469802e-14,True
316435_3_2439_1_peg,-3.246878e-12,93.91435,-3.457275e-14,True
316435_3_597_8_peg,-3.011597e-12,93.91435,-3.206748e-14,True
316435_3_2228_peg,-1.317574e-12,93.91435,-1.402952e-14,True
316435_3_3553_12_peg,-2.352810e-13,93.91435,-2.505272e-15,True
316435_3_1852_7_peg,-1.999888e-13,93.91435,-2.129481e-15,True


In [11]:
%%writefile ../tests/test_screen.py
"""Tests for lbpscreen.screen, on a tiny hand-made model."""
from cobra import Metabolite, Model, Reaction

import lbpscreen as lbp


def parallel_model():
    """a enters, becomes b by EITHER AB or AB2, then b -> biomass. Grows at 10."""
    a_e = Metabolite("a_e", compartment="e")
    b_c = Metabolite("b_c", compartment="c")
    bio_c = Metabolite("biomass_c", compartment="c")
    m = Model("parallel")

    def rxn(rid, stoich, lb=0, ub=1000, gpr=""):
        r = Reaction(rid, lower_bound=lb, upper_bound=ub)
        r.add_metabolites(stoich)
        r.gene_reaction_rule = gpr
        return r

    m.add_reactions([
        rxn("EX_a_e_", {a_e: -1}, lb=-10),
        rxn("AB",  {a_e: -1, b_c: 1}, gpr="geneA"),    # one of two routes in
        rxn("AB2", {a_e: -1, b_c: 1}, gpr="geneA2"),   # the backup route
        rxn("BIO", {b_c: -1, bio_c: 1}, gpr="geneB"),  # only biomass route: essential
        rxn("EX_biomass_c_", {bio_c: -1}),
    ])
    m.objective = "BIO"
    return m


def test_growth_table():
    table = lbp.growth_table({"x": parallel_model()})
    assert table.loc["x", "status"] == "optimal"
    assert abs(table.loc["x", "growth"] - 10) < 1e-6


def test_flux_range():
    m = parallel_model()
    fva = lbp.flux_range(m, ["BIO", "AB"], fraction=1.0)
    assert abs(fva.loc["BIO", "minimum"] - 10) < 1e-6   # biomass flux is forced to 10
    assert bool(fva.loc["BIO", "fixed"])                # no freedom
    assert not bool(fva.loc["AB", "fixed"])             # AB varies 0..10 (AB2 covers rest)


def test_essential_genes():
    ess = lbp.essential_genes(parallel_model())
    assert bool(ess.loc["geneB", "essential"])          # only biomass route
    assert not bool(ess.loc["geneA", "essential"])      # AB2 is a backup
    assert not bool(ess.loc["geneA2", "essential"])

Writing ../tests/test_screen.py


In [12]:
counts = {}
for label, model in models.items():
    ess = lbp.essential_genes(model)
    counts[label] = {"genes": len(ess), "essential": int(ess["essential"].sum())}
pd.DataFrame(counts).T

,genes,essential
A_muciniphila_BAA835,587,215
E_coli_Nissle1917,1457,136
E_faecalis_MMH594,1140,142
E_lenta_DSM2243,690,153
